# Jakarta live air quality — notebook lab

Use this notebook for **data fetch**, **tables**, and an **interactive Jakarta map** (ISPU color-coded stations from [udara.jakarta.go.id](https://udara.jakarta.go.id)).

**Multi-agent CrewAI report:** run **`main_v1.py` in the terminal**, not here — see [`docs/workshop-attendee-guide.md`](../docs/workshop-attendee-guide.md) (CAI web terminals often truncate long agent output).

**Cloudera AI:** Python **3.11**, project env `OPENAI_API_KEY` only needed for the terminal script. Faster installs: [`docs/cai-workshop-dependencies.md`](../docs/cai-workshop-dependencies.md).

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "main_v1.py").exists() and (ROOT / "AirAware" / "main_v1.py").exists():
    ROOT = ROOT / "AirAware"
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Working directory:", ROOT)

In [ ]:
from agent_tools.workshop_install import ensure_minimal

ensure_minimal(ROOT)

## 2. Fetch live SPKU snapshot

In [ ]:
from agent_tools.jakarta_air_sources import fetch_jakarta_realtime_dataframe

long_df = fetch_jakarta_realtime_dataframe(
    ["Jakarta, Indonesia"], ["pm25", "ispu"], include_open_meteo=False
)
print(f"Long-format rows: {len(long_df)}")
long_df.head(8)

In [ ]:
from agent_tools.jakarta_viz import fetch_station_snapshot_table

stations, updated_at = fetch_station_snapshot_table()
print(f"Geolocated stations: {len(stations)}")
print(f"Udara update time: {updated_at}")
stations.sort_values("ispu", ascending=False).head(10)

## 3. Map — ISPU by station

Circle color follows the **ISPU** band (green = good → red/purple = worse). Click markers for station name and value.

If you see `No module named folium`, re-run **section 1 setup** (CAI often has pandas preinstalled but not folium).

In [ ]:
from agent_tools.workshop_install import ensure_minimal
from agent_tools.jakarta_viz import build_ispu_map

ensure_minimal(ROOT)  # installs folium if the base image only had pandas/requests
m = build_ispu_map(stations, updated_at=updated_at)
m

In [ ]:
# Optional: worst / best ISPU today
from IPython.display import display

cols = ["station", "kecamatan", "ispu", "dominant_metric", "dominant_value"]
print("Highest ISPU:")
display(stations.nlargest(5, "ispu")[cols])
print("Lowest ISPU:")
display(stations.nsmallest(5, "ispu")[cols])

## 4. CrewAI agent report — use the terminal

Open **Workbench → Terminal** in the same CAI project and run:

```bash
cd AirAware
export OPENAI_API_KEY='your-key'   # or set in project environment variables
python3 main_v1.py --quiet -o report.md \
  --user-input "Snapshot for Jakarta: highlight worst and best ISPU stations today."
cat report.md
```

Full instructions: **`docs/workshop-attendee-guide.md`**.

In [ ]:
# Print the exact command for this session path (copy from output)
print(f"cd {ROOT}")
print(
    'python3 main_v1.py --quiet -o report.md '
    '--user-input "Snapshot for Jakarta: highlight worst and best ISPU stations today."'
)